# LangGraph State Machine

This notebook implements the core agent logic using LangGraph state machine patterns.

In [0]:
%pip install --upgrade numpy<2.0 langgraph langsmith clickhouse-connect --quiet
dbutils.library.restartPython()

In [0]:
"""Configure LangSmith for distributed tracing

LangSmith provides observability into LangGraph agent execution:
- Trace each node's execution time and outputs
- Debug state transitions and routing decisions
- Monitor LLM calls and token usage

Configuration:
- LANGCHAIN_TRACING_V2: Enables tracing
- LANGCHAIN_PROJECT: Groups traces into a named project
- LANGCHAIN_ENDPOINT: LangSmith API endpoint
- LANGCHAIN_API_KEY: Authentication for LangSmith
"""

import os
from langsmith import traceable

# Configure LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "langchain-production-agent"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = dbutils.secrets.get(scope="langsmith", key="api_key")

print("✅ LangSmith tracing configured")
print(f"   Project: {os.environ['LANGCHAIN_PROJECT']}")
print(f"   Endpoint: {os.environ['LANGCHAIN_ENDPOINT']}")
print(f"   Tracing enabled: {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"   API Key: {'*' * 20} (configured)")

In [0]:
"""Step 1: Define the Agent's Architecture

This agent implements a three-node architecture for clinical data queries:

1. The Planner (Router): Analyzes requests to route to appropriate tables
   - clinical_encounters: Visit and encounter data
   - patient_registry: Patient demographic information

2. The Analyst (SQL Generator): Converts natural language to Spark SQL
   - Targets clinical_dev.analytics_agent schema
   - Generates optimized SELECT queries

3. The Validator (Guardrail): Safety layer for SQL execution
   - Ensures only SELECT statements are executed
   - Blocks destructive operations (DROP, DELETE, UPDATE, etc.)
"""

import os
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

# Define the agent's state structure
class AgentState(TypedDict):
    """State container for the clinical data agent.
    
    Attributes:
        user_query: Natural language question from user
        generated_sql: AI-generated Spark SQL query
        is_safe: Whether the SQL passed validation checks
        final_answer: Formatted result to return to user
    """
    user_query: str
    generated_sql: Optional[str]
    is_safe: Optional[bool]
    final_answer: Optional[str]

# Initialize the state graph
workflow = StateGraph(AgentState)

print("✅ Agent State and Framework initialized")
print(f"   State fields: {list(AgentState.__annotations__.keys())}")
print(f"   Ready for node definitions")

In [0]:
"""The Analyst Node: Generating SQL

This node is the "Translator." It takes the user's plain English and turns it 
into Spark SQL to query the clinical data tables.

Available tables:
- clinical_dev.analytics_agent.clinical_encounters (visit/encounter data)
- clinical_dev.analytics_agent.patient_registry (patient demographics)
"""

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# Initialize Databricks workspace client
w = WorkspaceClient()

@traceable(name="Clinical_SQL_Analyst")
def sql_analyst_node(state: AgentState) -> dict:
    """Generate Spark SQL from natural language query.
    
    Args:
        state: Current agent state containing user_query
        
    Returns:
        Dictionary with generated_sql key
    """
    query = state["user_query"]
    
    system_prompt = """
    You are a SQL expert specializing in clinical data analysis using Spark SQL.
    
    Available tables in clinical_dev.analytics_agent:
    1. clinical_encounters: Visit and encounter data (ENCOUNTERCLASS, START, STOP, PATIENT, etc.)
    2. patient_registry: Patient demographics (BIRTHDATE, GENDER, RACE, ETHNICITY, etc.)
    
    Guidelines:
    - Write clean, optimized Spark SQL
    - Use appropriate JOINs when data spans multiple tables
    - Include meaningful column aliases
    - Add WHERE clauses for filtering when appropriate
    - Return ONLY the SQL code, no explanations or markdown formatting
    """
    
    try:
        # Call Databricks Foundation Model API
        # Using Meta Llama 3.3 70B - excellent for SQL generation
        response = w.serving_endpoints.query(
            name="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
                ChatMessage(role=ChatMessageRole.USER, content=query)
            ],
            max_tokens=512,
            temperature=0.0
        )
        
        generated_sql = response.choices[0].message.content.strip()
        
        # Remove markdown code blocks if present
        if generated_sql.startswith("```"):
            lines = generated_sql.split("\n")
            # Skip first line if it's just ```sql or ```
            start_idx = 1
            # Find last line with ```
            end_idx = len(lines)
            for i in range(len(lines) - 1, -1, -1):
                if lines[i].strip() == "```":
                    end_idx = i
                    break
            generated_sql = "\n".join(lines[start_idx:end_idx])
        
        return {"generated_sql": generated_sql.strip()}
        
    except Exception as e:
        print(f"❌ Error generating SQL: {str(e)}")
        return {"generated_sql": None}

# Add node to the workflow graph
workflow.add_node("analyst", sql_analyst_node)

# Set as the entry point for the agent
workflow.set_entry_point("analyst")

print("✅ Analyst node added and set as entry point")
print("   - Uses Meta Llama 3.3 70B Instruct via SDK")
print("   - Handles natural language to SQL translation")
print("   - Supports clinical_encounters and patient_registry tables")

In [0]:
"""The Validator Node: SQL Safety Guardrail

This node implements a deterministic safety layer that inspects AI-generated SQL
before execution. It blocks any destructive operations to ensure read-only access.

Safety Checks:
1. Forbidden keywords: DROP, DELETE, UPDATE, INSERT, TRUNCATE, ALTER
2. Must start with SELECT statement
3. Uses hard-coded Python logic (not AI-based)

Why This Matters:
- Deterministic Safety: Not relying on AI "feelings" - using explicit rules
- Defense in Depth: Multiple validation layers protect against destructive operations
- State Preservation: Only updates is_safe flag, leaves generated_sql unchanged
"""

import re

@traceable(name="Safety_Validator")
def sql_validator_node(state: AgentState) -> dict:
    """Validate SQL query for safety before execution.
    
    Args:
        state: Current agent state containing generated_sql
        
    Returns:
        Dictionary with is_safe boolean flag
    """
    # Handle None or empty SQL
    if not state.get("generated_sql"):
        print("❌ Safety Check: FAILED (No SQL generated)")
        return {"is_safe": False}
    
    sql = state["generated_sql"].upper()
    
    # Define forbidden keywords for read-only access
    forbidden_keywords = [
        "DROP", "DELETE", "UPDATE", "INSERT", "TRUNCATE", 
        "ALTER", "CREATE", "REPLACE", "MERGE"
    ]
    
    # Check 1: Scan for forbidden keywords
    # Use word boundaries to avoid false positives (e.g., "DROPPED" in column names)
    for keyword in forbidden_keywords:
        pattern = r'\b' + keyword + r'\b'
        if re.search(pattern, sql):
            print(f"❌ Safety Check: FAILED (Found forbidden keyword: {keyword})")
            return {"is_safe": False}
    
    # Check 2: Must start with SELECT
    if not sql.strip().startswith("SELECT"):
        print("❌ Safety Check: FAILED (Does not start with SELECT)")
        return {"is_safe": False}
    
    # All checks passed
    print("✅ Safety Check: PASSED")
    return {"is_safe": True}

# Add validator node to the workflow graph
workflow.add_node("validator", sql_validator_node)

# Connect analyst to validator: SQL generation → Safety check
workflow.add_edge("analyst", "validator")

print("✅ Validator node added and connected")
print("   - Implements deterministic safety checks")
print("   - Blocks destructive SQL operations")
print("   - Ensures read-only database access")

In [0]:
"""Step 4: Define Router Function

The router function implements conditional branching based on safety validation.
It will be used to route the workflow after SQL validation:
- If SQL passes validation → continue to execution
- If SQL fails validation → stop immediately

Note: The actual graph edges and compilation happen in the next cell
after all nodes (including executor and summarizer) are defined.
"""

def safety_router(state: AgentState) -> str:
    """Route the workflow based on SQL safety validation.
    
    Args:
        state: Current agent state containing is_safe flag
        
    Returns:
        'continue' if SQL is safe, 'stop' if unsafe
    """
    if state.get("is_safe"):
        return "continue"
    else:
        return "stop"

print("✅ Router function defined")
print("   - Returns 'continue' for safe SQL")
print("   - Returns 'stop' for unsafe SQL")
print("   - Ready to add executor and summarizer nodes")

In [0]:
"""Test the Complete Agent Workflow

This test validates the entire state machine pipeline:
1. Natural language query processing
2. SQL generation via Databricks DBRX
3. Safety validation with deterministic rules
4. Conditional routing based on validation results

Expected Behavior:
- Analyst generates a SELECT statement targeting clinical_encounters
- Validator passes the query (no forbidden keywords detected)
- Router continues to END (safe execution path)
"""

from langsmith import trace
import time

# Define the clinical question to test
input_query = "How many unique patients had inpatient encounters in 2023?"

# Initialize state with only required fields
initial_state = {
    "user_query": input_query
}

print("="*70)
print(f"🚀 AGENT EXECUTION TEST")
print("="*70)
print(f"\n📝 User Query: {input_query}\n")

try:
    # Execute with explicit LangSmith tracing context
    with trace(name="Clinical_Agent_Workflow", 
               project_name="langchain-production-agent",
               inputs={"query": input_query}) as run_context:
        
        # Execute the compiled state machine
        final_output = app.invoke(initial_state)
        
        # Set outputs in trace
        run_context.end(outputs=final_output)
    
    # Small delay to ensure trace is sent
    time.sleep(1)
    
    # Display results with clear formatting
    print("="*70)
    print("📊 EXECUTION RESULTS")
    print("="*70)
    
    # Show generated SQL
    generated_sql = final_output.get('generated_sql')
    if generated_sql:
        print(f"\n✅ Generated SQL:")
        print("-" * 70)
        print(generated_sql)
        print("-" * 70)
    else:
        print(f"\n❌ No SQL generated")
    
    # Show safety status
    is_safe = final_output.get('is_safe', False)
    safety_status = "✅ PASSED" if is_safe else "❌ FAILED"
    print(f"\n🛡️  Safety Validation: {safety_status}")
    
    # Summary
    print(f"\n📈 Workflow Status: {'SUCCESS' if is_safe else 'BLOCKED'}")
    print("="*70)
    print("\n✅ Trace sent to LangSmith project: langchain-production-agent")
    
except Exception as e:
    print(f"\n❌ ERROR during execution: {str(e)}")
    print(f"   Type: {type(e).__name__}")
    import traceback
    print(f"\n{traceback.format_exc()}")

In [0]:
"""Step 5: SQL Executor and Clinical Summarizer

Adding two final nodes to complete the agent workflow:

1. Executor Node: Runs the validated SQL query against Databricks
   - Executes safe queries that passed validation
   - Returns raw query results
   - Handles execution errors gracefully

2. Summarizer Node: Transforms raw data into professional clinical insights
   - Uses LLM to generate executive summaries
   - Provides actionable insights for healthcare operations
   - Formats findings in clear, concise bullet points
"""

from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
from databricks.sdk.service.sql import StatementState
import time

@traceable(name="SQL_Executor")
def sql_executor_node(state: AgentState) -> dict:
    """Execute validated SQL query and return results.
    
    Args:
        state: Current agent state with validated SQL
        
    Returns:
        Dictionary with final_answer containing query results
    """
    sql = state.get("generated_sql")
    
    if not sql:
        return {"final_answer": "No SQL to execute"}
    
    try:
        # Use Databricks SQL Statement Execution API
        # This avoids numpy/pandas compatibility issues
        statement = w.statement_execution.execute_statement(
            warehouse_id="e7ab584a1feb58ef",  # Serverless Starter Warehouse
            statement=sql,
            wait_timeout="30s"
        )
        
        # Check if execution succeeded (use enum, not string)
        if statement.status and statement.status.state == StatementState.SUCCEEDED:
            # Get result data
            if statement.result and statement.result.data_array:
                # Format results
                columns = [col.name for col in statement.manifest.schema.columns] if statement.manifest else []
                rows = statement.result.data_array
                
                if len(rows) == 0:
                    result_str = "Query executed successfully but returned no results."
                else:
                    # Build formatted output
                    result_str = "  ".join(columns) + "\n"
                    result_str += "-" * (len(result_str) - 1) + "\n"
                    
                    for row in rows[:100]:  # Limit to 100 rows
                        row_values = [str(val) if val is not None else "NULL" for val in row]
                        result_str += "  ".join(row_values) + "\n"
                
                return {"final_answer": result_str.strip()}
            else:
                return {"final_answer": "Query executed successfully with no result data."}
        else:
            error_msg = f"Query execution failed with status: {statement.status.state if statement.status else 'UNKNOWN'}"
            return {"final_answer": error_msg}
            
    except Exception as e:
        error_msg = f"Query execution failed: {str(e)}"
        print(f"❌ {error_msg}")
        return {"final_answer": error_msg}

@traceable(name="Clinical_Summarizer")
def clinical_summarizer_node(state: AgentState) -> dict:
    """Generate professional clinical insights from raw query results.
    
    Args:
        state: Current agent state with query results
        
    Returns:
        Dictionary with final_answer containing formatted summary
    """
    query = state.get("user_query", "")
    raw_data = state.get("final_answer", "")
    
    # If there was an execution error, return it directly
    if "failed" in raw_data.lower() or "error" in raw_data.lower():
        return {"final_answer": raw_data}
    
    system_prompt = """
    You are a Senior Healthcare Data Analyst specializing in clinical operations.
    
    Your role:
    - Transform raw SQL query results into clear, actionable insights
    - Use bullet points for key metrics and findings
    - Provide context on what the data means for hospital operations
    - Be concise and assertive in your analysis
    - Focus on operational impact and clinical significance
    """
    
    user_prompt = f"""
    User Question: {query}
    
    Query Results:
    {raw_data}
    
    Provide a professional summary of these findings.
    """
    
    try:
        # Call LLM with proper ChatMessage format
        response = w.serving_endpoints.query(
            name="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
                ChatMessage(role=ChatMessageRole.USER, content=user_prompt)
            ],
            max_tokens=512,
            temperature=0.3  # Slightly creative but still focused
        )
        
        summary = response.choices[0].message.content.strip()
        return {"final_answer": summary}
        
    except Exception as e:
        print(f"❌ Error generating summary: {str(e)}")
        # Fall back to raw data if summarization fails
        return {"final_answer": raw_data}

# Add executor and summarizer nodes to the workflow
workflow.add_node("executor", sql_executor_node)
workflow.add_node("summarizer", clinical_summarizer_node)

# Update the router to send safe queries to executor
workflow.add_conditional_edges(
    "validator",
    safety_router,
    {
        "continue": "executor",  # Safe SQL → Execute
        "stop": END              # Unsafe SQL → Stop immediately
    }
)

# Connect executor to summarizer, then to end
workflow.add_edge("executor", "summarizer")
workflow.add_edge("summarizer", END)

# Re-compile the graph with new nodes
app = workflow.compile()

print("✅ Executor and Summarizer nodes added")
print("   - SQL Executor: Uses Databricks SQL Statement Execution API")
print("   - Warehouse: Serverless Starter Warehouse (e7ab584a1feb58ef)")
print("   - Clinical Summarizer: Generates insights with Meta Llama 3.3 70B")
print("   - Updated graph: Analyst → Validator → Router → Executor → Summarizer → END")
print("   - Ready for complete end-to-end testing")

In [0]:
"""Test the Complete 5-Node Agent Workflow

This test validates the entire state machine pipeline including execution and summarization:
1. Natural language query processing (Analyst)
2. SQL generation via Meta Llama 3.3 70B (Analyst)
3. Safety validation with deterministic rules (Validator)
4. SQL execution on Databricks SQL warehouse (Executor)
5. Clinical insight generation (Summarizer)
"""

from langsmith import trace
import time

# Define the clinical question to test
input_query = "How many unique patients had inpatient encounters in 2023?"

# Initialize state with only required fields
initial_state = {
    "user_query": input_query
}

print("="*70)
print(f"🚀 COMPLETE AGENT WORKFLOW TEST")
print("="*70)
print(f"\n📝 User Query: {input_query}\n")

try:
    # Execute with explicit LangSmith tracing context
    with trace(name="Complete_Clinical_Agent_Workflow", 
               project_name="langchain-production-agent",
               inputs={"query": input_query}) as run_context:
        
        # Execute the compiled state machine
        final_output = app.invoke(initial_state)
        
        # Set outputs in trace
        run_context.end(outputs=final_output)
    
    # Small delay to ensure trace is sent
    time.sleep(1)
    
    # Display results with clear formatting
    print("="*70)
    print("📊 WORKFLOW EXECUTION RESULTS")
    print("="*70)
    
    # Step 1: Show generated SQL
    generated_sql = final_output.get('generated_sql')
    if generated_sql:
        print(f"\n✅ Step 1 - SQL Generation:")
        print("-" * 70)
        print(generated_sql)
        print("-" * 70)
    else:
        print(f"\n❌ Step 1 - SQL Generation: FAILED")
    
    # Step 2: Show safety status
    is_safe = final_output.get('is_safe', False)
    safety_status = "✅ PASSED" if is_safe else "❌ FAILED"
    print(f"\n✅ Step 2 - Safety Validation: {safety_status}")
    
    # Step 3 & 4: Show final answer (after execution and summarization)
    final_answer = final_output.get('final_answer', '')
    if final_answer:
        print(f"\n✅ Steps 3-5 - Execution & Summarization:")
        print("-" * 70)
        print(final_answer)
        print("-" * 70)
    else:
        print(f"\n⚠️  Steps 3-5: No final answer generated")
    
    # Summary
    print(f"\n📈 Workflow Status: {'COMPLETE SUCCESS' if is_safe and final_answer else 'PARTIAL SUCCESS'}")
    print("="*70)
    print("\n✅ Trace sent to LangSmith project: langchain-production-agent")
    print("\n🔍 View traces at: https://smith.langchain.com")
    
except Exception as e:
    print(f"\n❌ ERROR during execution: {str(e)}")
    print(f"   Type: {type(e).__name__}")
    import traceback
    print(f"\n{traceback.format_exc()}")

In [0]:
"""🎯 SHOWCASE TEST: Multi-Year Trend Analysis

This test demonstrates the agent's ability to handle complex analytical queries:
- Multi-year comparison (2023 vs 2024)
- Trend calculation and interpretation
- Strategic recommendations for hospital leadership

Expected Workflow:
1. Analyst: Generate SQL with GROUP BY year and COUNT aggregation
2. Validator: Verify SQL safety (SELECT only)
3. Executor: Run query on SQL warehouse
4. Summarizer: Calculate trends and provide leadership insights
"""

from langsmith import trace
import time

# Define the complex analytical question
input_query = "Compare the total number of inpatient encounters between 2023 and 2024. What is the trend, and what should the hospital leadership focus on?"

# Initialize state
initial_state = {
    "user_query": input_query
}

print("="*70)
print(f"🎯 SHOWCASE TEST: Multi-Year Trend Analysis")
print("="*70)
print(f"\n📝 Executive Question: {input_query}\n")

try:
    # Execute with LangSmith tracing
    with trace(name="Showcase_Trend_Analysis", 
               project_name="langchain-production-agent",
               inputs={"query": input_query}) as run_context:
        
        # Execute the complete state machine
        final_output = app.invoke(initial_state)
        
        # Set outputs in trace
        run_context.end(outputs=final_output)
    
    # Small delay to ensure trace is sent
    time.sleep(1)
    
    # Display results
    print("="*70)
    print("📊 SHOWCASE RESULTS")
    print("="*70)
    
    # Step 1: Generated SQL
    generated_sql = final_output.get('generated_sql')
    if generated_sql:
        print(f"\n🔬 Step 1 - SQL Generation (Analyst Node):")
        print("-" * 70)
        print(generated_sql)
        print("-" * 70)
    else:
        print(f"\n❌ Step 1 - SQL Generation: FAILED")
    
    # Step 2: Safety validation
    is_safe = final_output.get('is_safe', False)
    safety_icon = "✅" if is_safe else "❌"
    print(f"\n{safety_icon} Step 2 - Safety Validation (Validator Node): {'PASSED' if is_safe else 'FAILED'}")
    
    # Steps 3-5: Execution and Clinical Insights
    final_answer = final_output.get('final_answer', '')
    if final_answer:
        print(f"\n📈 Steps 3-5 - Execution & Clinical Analysis:")
        print("-" * 70)
        print(final_answer)
        print("-" * 70)
    else:
        print(f"\n⚠️  Steps 3-5: No final answer generated")
    
    # Final summary
    workflow_status = "🏆 COMPLETE SUCCESS" if is_safe and final_answer else "⚠️  PARTIAL SUCCESS"
    print(f"\n{workflow_status}")
    print("="*70)
    print("\n✅ All traces sent to LangSmith: https://smith.langchain.com")
    print("\n💡 Agent demonstrated:")
    print("   • Complex SQL generation (GROUP BY, multi-year comparison)")
    print("   • Deterministic safety validation")
    print("   • Databricks SQL warehouse execution")
    print("   • Strategic clinical insights with trend analysis")
    
except Exception as e:
    print(f"\n❌ ERROR during showcase test: {str(e)}")
    print(f"   Type: {type(e).__name__}")
    import traceback
    print(f"\n{traceback.format_exc()}")

In [0]:
"""Production Test: Agent Execution with Dual Observability

This test demonstrates the complete production workflow:
1. Execute clinical agent with complex query
2. Log execution trace to ClickHouse for observability (best-effort)
3. Log execution trace to LangSmith for distributed tracing (primary)
4. Display clinical insights with performance metrics

Note: ClickHouse logging may fail due to numpy binary incompatibility on serverless
clusters. This is a known limitation and fails gracefully without blocking the workflow.
LangSmith provides full observability as the primary tracing solution.
"""

import time
from typing import Dict, Any
from langsmith import trace
from clickhouse_connect import get_client

def log_agent_run(state: Dict[str, Any], start_time: float) -> None:
    """
    Log agent execution trace to ClickHouse for observability (best-effort).
    
    Args:
        state: AgentState dictionary containing query, response, SQL, and safety info
        start_time: Unix timestamp when agent execution started
        
    Note:
        May fail on serverless clusters due to numpy binary incompatibility.
        Failure is non-blocking - LangSmith provides primary observability.
    """
    try:
        # Calculate latency in milliseconds
        latency_ms = (time.time() - start_time) * 1000
        
        # Validate required fields
        if not state.get("user_query") or not state.get("final_answer"):
            print("⚠️  Skipping ClickHouse log: missing required fields")
            return
        
        # Connect to ClickHouse using Databricks secrets
        client = get_client(
            host=dbutils.secrets.get(scope="clickhouse", key="host"),
            port=int(dbutils.secrets.get(scope="clickhouse", key="port")),
            username=dbutils.secrets.get(scope="clickhouse", key="user"),
            password=dbutils.secrets.get(scope="clickhouse", key="password"),
            secure=True
        )
        
        # Prepare trace data
        data = [[
            state["user_query"],
            state["final_answer"],
            0,  # tokens_used placeholder
            state.get("generated_sql", "N/A"),
            1 if state.get("is_safe") else 0,
            latency_ms
        ]]
        
        # Insert trace into ClickHouse
        client.insert(
            'ai_logs',
            data,
            column_names=['prompt', 'response', 'tokens_used', 'generated_sql', 'is_safe', 'latency_ms']
        )
        
        # Confirmation with metrics
        safety_status = "✅ SAFE" if state.get("is_safe") else "❌ UNSAFE"
        print(f"✅ ClickHouse trace logged | Latency: {latency_ms:.2f}ms | Safety: {safety_status}")
        
    except Exception as e:
        # Known limitation: numpy binary incompatibility on serverless
        print(f"⚠️  ClickHouse logging skipped (known serverless limitation)")
        # Workflow continues - LangSmith provides observability
    finally:
        if 'client' in locals():
            client.close()

print("="*70)
print("🎯 PRODUCTION TEST: Agent with Dual Observability")
print("="*70)

# Define clinical query
query = "What was the inpatient admission count for Q4 2024?"
print(f"\n📝 Query: {query}\n")

try:
    # Start performance timer
    start_time = time.time()
    
    # Execute agent with LangSmith tracing
    with trace(name="Production_Agent_with_Dual_Observability",
               project_name="langchain-production-agent",
               inputs={"query": query}) as run_context:
        
        # Run the clinical agent workflow
        final_state = app.invoke({"user_query": query})
        
        # Set trace outputs
        run_context.end(outputs=final_state)
    
    # Calculate execution time
    execution_time = (time.time() - start_time) * 1000
    
    # Log to ClickHouse (best-effort, may fail gracefully)
    log_agent_run(final_state, start_time)
    
    # Display results
    print("\n" + "="*70)
    print("📊 CLINICAL REPORT")
    print("="*70)
    
    # Show safety status
    is_safe = final_state.get('is_safe', False)
    safety_icon = "✅" if is_safe else "❌"
    print(f"\n{safety_icon} Safety Validation: {'PASSED' if is_safe else 'FAILED'}")
    
    # Show generated SQL
    if final_state.get('generated_sql'):
        print(f"\n🔬 Generated SQL:")
        print("-" * 70)
        print(final_state['generated_sql'])
        print("-" * 70)
    
    # Show clinical insights
    final_answer = final_state.get('final_answer', 'No answer generated')
    print(f"\n📝 Clinical Insights:")
    print("-" * 70)
    print(final_answer)
    print("-" * 70)
    
    # Performance metrics
    print(f"\n⏱️  Execution Time: {execution_time:.2f}ms")
    print(f"✅ LangSmith trace: https://smith.langchain.com/o/c1ccb7ff-4c74-411e-b35b-f23497b486de/projects/p/langchain-production-agent")
    print("\n" + "="*70)
    print("🏆 PRODUCTION TEST COMPLETE")
    print("="*70)
    print("\n💡 Observability Status:")
    print("   ✅ LangSmith: Full distributed tracing active")
    print("   ⚠️  ClickHouse: Best-effort logging (may skip on serverless)")
    
except Exception as e:
    print(f"\n❌ ERROR during production test: {str(e)}")
    print(f"   Type: {type(e).__name__}")
    import traceback
    print(f"\n{traceback.format_exc()}")